In [1]:
import pandas as pd

# Load core files
books = pd.read_csv("data/goodbooks-10k/books.csv")
book_tags = pd.read_csv("data/goodbooks-10k/book_tags.csv")
tags = pd.read_csv("data/goodbooks-10k/tags.csv")
ratings_books = pd.read_csv("data/goodbooks-10k/ratings.csv")
to_read = pd.read_csv("data/goodbooks-10k/to_read.csv")

# Step 1: Join book_tags with tags to get tag names
book_tags_named = book_tags.merge(tags, on="tag_id")

# Step 2: Merge with books using goodreads_book_id
books_enriched = books.merge(book_tags_named, on="goodreads_book_id", how="left")

# Step 3: Compute average rating per book_id
avg_ratings = ratings_books.groupby("book_id")["rating"].mean().reset_index()

# Step 4: Merge ratings into enriched books
books_enriched = books_enriched.merge(avg_ratings, on="book_id", how="left")

# Step 5: Compute how many user marked each book as to-read
to_read_count = to_read.groupby("book_id").size().reset_index(name="to_read_count")

# Step 6: Merge read_count into enriched books
books_enriched = books_enriched.merge(to_read_count, on="book_id", how="left")

# Optional: clean up NaN values
books_enriched["rating"] = books_enriched["rating"].fillna("No rating")
books_enriched["tag_name"] = books_enriched["tag_name"].fillna("No tags")

# Group tags into a list per book
books_grouped = books_enriched.groupby(
    ["book_id", "title", "authors", "original_publication_year", "rating", "to_read_count"]
)["tag_name"].apply(list).reset_index()

In [2]:
# Total rows
total_rows = len(books_grouped)

# Count missing and non-missing per column
missing_count = books_grouped.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)

                           non_missing  missing  missing_percent
book_id                           9965        0              0.0
title                             9965        0              0.0
authors                           9965        0              0.0
original_publication_year         9965        0              0.0
rating                            9965        0              0.0
to_read_count                     9965        0              0.0
tag_name                          9965        0              0.0
